# run_experiment

This notebook runs one complete experiment from a settings file. It works on Google
Colab and on a rented GPU machine; the only difference between the two is the
settings file that is named.

Follow these steps:

1. Edit the first code cell only. Set `SETTINGS_FILE` to the chosen experiment and
   `HUB_REPO` to a Hugging Face repository you can write to.
2. Run all cells in order.
3. To run a further experiment, change `SETTINGS_FILE` and run again from
   **Run the experiment** onwards. The model is already downloaded and the
   environment is already installed, so each further run costs only its own
   training time.

On a rented machine that is deleted when it expires, run the save cell after every
experiment rather than only at the end. Everything left on the machine is lost when
it shuts down.

In [ ]:
USE_TEST_CONFIG = False         # True: the CPU smoke settings in src/configs/test.py
SETTINGS_FILE = "configs/default.json"

# The Hugging Face repository where finished runs are saved, so that they survive the
# machine. You must be able to write to it: its owner must be the account, or an
# organisation of the account, that your token belongs to.
HUB_REPO = "CHANGE-ME/bdp-plunkett-qwen3"

# Settings files available:
#   configs/default.json             THE DEFAULT: Plunkett on Qwen3-0.6B, 4-bit, fp16 (Colab T4)
#   configs/atkinson.json            the same, with 3000 steps and no introspection training
#   configs/linear-4b_20gb.json      Qwen3-4B, bf16, no compression, gradient checkpointing
#   configs/linear-8b_20gb.json      Qwen3-8B, bf16, 4-bit, gradient checkpointing
#   configs/a4_8b.json               the constrained tradeoff at Qwen3-8B, 3000 steps
#
# The two linear-*_20gb files are sized for 20 GB of GPU memory. This is the amount
# that the GPU check below reports on one MIG slice of an NVIDIA Hopper card.

## Set up the machine

The cells below check for a GPU, download the code, install the required packages,
confirm that they can be imported, and ask for a Hugging Face token.

In [ ]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, use USE_TEST_CONFIG = True"

In [ ]:
# Clone the pipeline branch when the code is not already here, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", "pipeline", CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

In [ ]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

In [ ]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")


In [ ]:
# Confirm the GPU matches what the settings assume. bf16 needs Ampere or newer:
# the 20gb configs ask for it and the loader refuses rather than falling back.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 30:
        print(f"\nThis is a MIG slice of about {vram:.0f} GB, not a whole card.")
        print("Use the *_20gb configs for 4B and 8B. 14B will not fit here.")
    elif vram < 60:
        print(f"\nThis is a MIG slice of about {vram:.0f} GB. The *_20gb configs fit with room to spare.")
else:
    print("no CUDA device: only USE_TEST_CONFIG = True will run here")

In [ ]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only for the save cells; the public Qwen3 weights do not
# require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

owner = HUB_REPO.split("/")[0]
who = None
for attempt in range(3):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if who is None:
    print("no token: the run works, but saving refuses and nothing survives the machine")
else:
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print(f"Set HUB_REPO = \"{who['name']}/bdp-plunkett-qwen3\" in the first cell and re-run it.")


## Run the experiment

On a rented machine, run the experiments in the following order and save after each
one. The smallest experiment tests the GPU path, the estimator, the report reader and
the results writer within a few minutes. It is therefore the least expensive place to
discover a faulty setting.

    default  ->  linear-4b_20gb  ->  linear-8b_20gb

In [ ]:
from src.config import load
from src.pipeline import run

cfg = load(None if USE_TEST_CONFIG else SETTINGS_FILE, use_test=USE_TEST_CONFIG)
rows = run(cfg)

## Save the results off the machine

Results are small, so they should be uploaded often. Adapters are tens of megabytes
each, so pass `checkpoints=False` for a quick upload during a run.

In [ ]:
from src.persist import push_run

push_run(rows[-1]["run_id"], HUB_REPO)

In [ ]:
# Before you walk away: sweep everything still on disk, including any run that was
# interrupted part way. Results are appended as a run goes, so a partial run is
# still worth keeping.
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)